In [60]:
using GLMakie
using Printf
using LinearAlgebra
using FFTW

function prettyrecord(observable, fig, filename, frames; record_kw...)
    N = length(frames)
    t0 = time()
    record(fig, filename, 1:N; record_kw...) do i
        observable[] = frames[i]
        
        t = time() - t0
        total_time = N * t / i
        
        str = @sprintf "%ds / %ds (%d / %d)" t total_time i N
        nl = i == N ? "\n" : "\r"
        
        print(str * nl)
    end
    return nothing
end

prettyrecord (generic function with 1 method)

In [101]:
# Create the profiles
include("src/profile/profile.jl")
include("src/modes/modes.jl")
include("src/packet/packet.jl")

Nx = 256
Nz = 256

# Geometry
H = 1000.0
L = 1000.0
N² = 1e-4
Δb = N² * H / 100

# Nodes for plotting profiles
profile_xs = range(-L/2, L/2, Nx)
profile_zs = range(-H, 0, Nz)

profile_names = [
    "Linear",
    "Step",
]

profile_descriptions = [
    "Simple constant stratification profile",
    "Step change in buoyancy"
]

profiles = [
    LinearProfile(H, N²),
    PycnoclineProfile(0.2H, H, N², 9N², N², 0.1H)
]

modes = map(profile_names, profiles) do profile_name, profile
    println("Creating modes for $profile_name...")
    Modes(profile, Nx, Nz, L)
end

Creating modes for Linear...
Creating modes for Step...


2-element Vector{Modes{Float64}}:
 Modes{Float64}(LinearProfile{Float64}(1000.0, 0.0001), 256, 256, 1000.0, [0.0, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009000000000000001  …  -0.01, -0.009000000000000001, -0.008, -0.007, -0.006, -0.005, -0.004, -0.003, -0.002, -0.001], [0.0 0.0 … 0.0 0.0; 1.1625388889864296e-5 1.1624930953763762e-5 … 8.90862579666237e-6 8.90841972173498e-6; … ; -2.3250730644848856e-5 -2.324981477821763e-5 … -1.7817230382740556e-5 -1.7816818234357673e-5; -1.1625388889864296e-5 -1.1624930953763762e-5 … -8.90862579666237e-6 -8.90841972173498e-6], [0.011625396745709433 0.011624938808680587 … 0.008908629331768149 0.008908423256595441; 0.011625388889864295 0.011624930953763763 … 0.00890862579666237 0.00890841972173498; … ; 0.011625365322424428 0.011624907389108815 … 0.008908615191370277 0.008908409117178837; 0.011625388889864295 0.011624930953763763 … 0.00890862579666237 0.00890841972173498], [0.0010783366584971696 0.0021565121847485283 … 0.00215651218474

In [152]:
# Initial packet
θ = π-0.1
λ = 30.0
k = 2π / λ * sin(θ)
m = 2π / λ * cos(θ)
z = -500.0
δ = 60.0
T = 800 * 3600
N = 300
ts = range(0, T, N)

# Background stratification
profile_n = 2
profile = profiles[profile_n]

n = Observable(1)
t = @lift ts[$n]

title = @lift let 
    t_val = $t / 3600
    hr_val = @sprintf "%.1f" t_val
    λ_val = @sprintf "%.0f" λ
    θ_val = @sprintf "%.2f" θ
    δ_val = @sprintf "%.0f" δ
    name = profile_names[profile_n]
    L"\text{%$name}, \quad t = %$hr_val \, \text{hr}, \quad \lambda=%$λ_val\,\text{m}, \quad \theta=%$θ_val, \quad δ=%$δ_val\,\text{m}"
end

println("Calculating modal_decomposition")
gaussianpacket = GaussianPacket(0.0, z, k, m, δ)
modepacket = ModePacket(gaussianpacket, modes[profile_n])
ω = Ω(profile, 0.0, z, k, m)

println("Plotting")
fig = Figure(; size=(600, 600))
ax = Axis(fig[1, 1];
    xlabel = L"x / \text{m}",
    ylabel = L"z / \text{m}",
    title,
    aspect = DataAspect(),
    xticks = [-500, -250, 0, 250, 500],
    yticks = [-1000, -750, -500, -250, 0]
)

xs = range(-L/2, L/2, Nx)
zs = range(bottom(profile), top(profile), Nz)
ψs = @lift streamfunction(modepacket, $t)

colorrange = (-maximum(abs, ψs[]), maximum(abs, ψs[]))
heatmap!(ax, xs, zs, ψs; colormap=:balance, colorrange)
contour!(ax, [-L/2, L/2], zs, (x, z)->buoyancy(profile, z); levels=30, color=:black)
contour!(ax, [-L/2, L/2], zs, (x, z)->frequency(profile, z); levels=[ω], color=:red)

#println("Recording...")
#prettyrecord(n, fig, "linear.mp4", 1:N; px_per_unit=2, framerate=24)

fig

Calculating modal_decomposition
Plotting


In [159]:
t[] = 800 * 3600

2880000

In [105]:
fs = [frequency(profile, z) for z in zs]
lines(fs, zs)

In [106]:
dominant_frequency(modepacket)

0.0011721595082967314